# Your Loss

In stage 1 we have a 'model' that has adjustable weights and bias. However if we just start guessing random numbers to set them at, the model will be bad at predicting
which applicants will default.

To measure this we need to measure exactly how 'wrong' the model is. That is what we call **Loss**.


## Accuracy isn't enough

- Accuracy alone isn't enough. If a model outputs a probability p = 0.52 for a positive example (y=1), it rounds to 1, so its accuracy is 100%. If you tweak a weight slightly
and p becomes 0.53, the accuracy is still 100%

- The metric doesn't change at all so the model receives 0 feedback on if adjusting the weight helped or made things worse. The only feedback it will get is the 0.5 boundary:




## Solution

Instead of asking if the rounded guess is correct, we need something that asks how much probability did he model assign to the actual ground truth
This is where Loss comes in. It rewards the model for being confident and right but punishes it for being confident and wrong.

## Defining the Loss Function 

This part is called optimising a Cost Function. For binary classification, we use **Binary Cross-Entropy** (BCE)
The formula is as follows:

`L = -[y.log(p) + (1-y).log(1-p)]`

This measures the dissimilarity between 2 proabibility distributions: the true distribution (y) and the predicted distribution (p).

Further breakdown of the log function:

- As p -> 1 (the model approaches certainty of the truth), -log(p) -> 0 The penalty starts to decrease
- As p -> 0 (the model approaches certainty of falsehood), -log(p) -> infinity The penalty increases

This exponential penalty curve is what gives the loss function its smooth nature, every decimal place change
in p yields a differnet predictable change in **L**.


Here's how the penalty behaves when the true label (y) is 1:

| Model's probability | Loss |
|:--- |:---:|
| 0.99 | 0.01 |
| 0.5 | 0.69 |
|0.01 | 4.61 |

## Gradient

Training a model requires finding the derivative of our scoring metric to the weights $\frac{Score}{w}$. The derivative tells us the rate of change. 
"if I change w by a tiny amount, how much does my score change"

- Accuracy is a step function. If a threshold is set at 0.5, anything prediction a model makes above this or below is all labelled Class 0 or 1 respectively.
- This means that if a model's probability goes from 0.55 to 0.56, the class at the end is still the same and the accuracy metric hasn't changed.
- If you were to visulaise this on a graph, the accuracy would have flat plateuas with sudden jumps. It's derivativie is 0 almost everywhere. The model has
no signal telling it which way to move the weights to.


Because Binary Cross-Entropy is perfectly smooth and continuous, it forms a differentiable loss landscape.
Imagine a multidimensional terrain where lattitude and longitude are the weights (w1, w2, ...) and bias (b),
and the altitude is the total Loss (L).

![Accuracy vs BCE Graph](Code_Generated_Image.png)



Gradient Vector $\nabla \mathcal{L}$ 

The gradient in gradient is a vector (list of numbers) containing the partial derivatives of the loss function with respect to each parameter in the model.

**Update Rule**
The gradient vector points in the direction of the steepest ascent (the fastest way up to the top).
Since we want to minimise loss, we do the exact opposite of this: we subtract the gradient.

The parameters are updated iteratively using this formula: $$ \mathbf{w}_{\text{new}}=\mathbf{w}_{\text{old}}-\eta \nabla \mathcal{L} $$


Where $\eta$ is the learning rate (a hyperparameter that dictates how large of a step the model takes down the slope).


## Reference Point

With all weights at 0, every prediction is 0.5, and the loss is **-log(0.5) = log(2) = 0.693**. That's the score of a model that knows nothing. If your trained model's loss
isn't clearly below 0.693, it hasn't learned anything useful which is a handful sanity check to remember.


## Overflows and Infinity

If z = wx+b leads to 1000 the sigmoid functino will overflow whereas if p is exactly 0, then log(0) in the loss function goes towards negative infinity 
and the loss breaks.

The fix is to never compute p and then take its log. Instead, you compute the loss directly from the score z.

$$L = log(1 + {e^z}) - y.z $$

NumPy's `np.logaddexp(0, z)` computes $ log(1 + {e^z}) $ safely for any z, large or small. We can use the same trick for a stable sigmoid




In [ ]:
import numpy as np

def sigmoid(z):
    return np.exp(-np.logaddexp(0, -z))

def predict_probability(X, w, b):
    return sigmoid(X @ w + b)

def bec_loss(X, y, w, b):
    z = X @ w + b
    return np.mean(np.logaddexp(0, z) - y * z)


0.0


Let's try this on our previous example of vectors, labels and weights from stage 1: 

In [7]:
X = np.array([[0.9, 0.1, 0.4],
                [-0.5, 0.8, -0.2],
                [-0.8, -0.7, -0.3]])

w = np.array([2.0, -1.0, 0.5])
b = 0.0
y = np.array([1, 0, 0])

print(predict_probability(X, w, b))
print(bec_loss(X, y, w, b))

[0.86989153 0.13010847 0.2592251 ]
0.19294399872785473


And when the weights are 0, we should expect that the loss value is 0.693
As above this is because there is no indication of the strength or relationship of each feature to a decision.
The model will just predict a probabiliy of 0.5 and the loss function L = -log(p) will be roughly 0.693 
This indicates that the model knows nothing


In [9]:
X = np.array([[0.9, 0.1, 0.4],
                [-0.5, 0.8, -0.2],
                [-0.8, -0.7, -0.3]])

w = np.array([0.0, 0.0, 0.0])
b = 0.0
y = np.array([1, 0, 0])

print(predict_probability(X, w, b))
print(bec_loss(X, y, w, b))

[0.5 0.5 0.5]
0.6931471805599453


Now let's flip the labels

IF we flip the labels around for each of the vectors, we should see a sharp rise in the loss function 

In [11]:
X = np.array([[0.9, 0.1, 0.4],
                [-0.5, 0.8, -0.2],
                [-0.8, -0.7, -0.3]])

w = np.array([2.0, -1.0, 0.5])
b = 0.0
y = np.array([0, 1, 1])

print(predict_probability(X, w, b))
print(bec_loss(X, y, w, b))

[0.86989153 0.13010847 0.2592251 ]
1.8096106653945212
